# Crossmatch and vetting

**Goal:** run the real search (no fakes) on a **validation** night, separate known asteroids from unknown tracklets, and
learn how to vet a candidate before believing it. The test night stays untouched until the final phase.

Read `docs/crossmatch_vetting.md` alongside this notebook.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from asthunt.config.load_config import load_config
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.acquire.query_ztf_metadata import query_ztf_metadata
from asthunt.features.compute_tracklet_features import FEATURE_COLUMNS
from asthunt.models.load_model_bundle import load_model_bundle
from asthunt.pipeline.run_night_pipeline import run_night_pipeline

project = load_config("project"); paths = project["paths"]
configs = {name: load_config(name) for name in ["acquire", "inject", "detect", "link"]}
acquire = configs["acquire"]


def unit_of(night):
    """The original quadrant (configs/acquire.yaml: field, ccd, quadrant) on one night."""
    return {"field": acquire["field"], "ccd": acquire["ccd"], "quadrant": acquire["quadrant"], "night": night}


metadata = load_unit_metadata(unit_of(None), paths["raw"])
device = "mps" if torch.backends.mps.is_available() else "cpu"

# One model bundle = one version of the whole system (CNN + tracklet classifier + thresholds)
bundle = load_model_bundle(paths["models"], "v1")
realbogus_model, tracklet_model, thresholds = bundle["realbogus_model"], bundle["tracklet_model"], bundle["thresholds"]
realbogus_file = tracklet_file = paths["models"] / "v1"
print("bundle", bundle["info"]["version"], thresholds)

## 1. Search a validation night, with no fakes planted

In [ ]:
night = acquire["nights"]["validation"][1]
rng = np.random.default_rng([project["seed"], int(night.replace("-", ""))])
result = run_night_pipeline(unit_of(night), metadata, configs, bundle, paths, rng, device)
tracklets, candidates = result["tracklets"], result["candidates"]
print(f"{night}: {len(result['detections'])} detections -> {len(tracklets)} tracklets -> "
      f"{(tracklets['known_name'] != '').sum()} known asteroids, {len(candidates)} unknown candidates")

## 2. Known asteroids: our sanity check against reality
Every tracklet that follows a SkyBoT object is a **real asteroid we found on our own**. Did the pipeline find the ones
it should have (bright enough)?

In [ ]:
known_tracklets = tracklets[tracklets["known_name"] != ""]
print(known_tracklets[["known_name", "n_detections", "rate_arcsec_per_hour", "mag_mean", "tracklet_score"]].round(2).to_string())

frames = result["detections"]["exposure"].unique()
known = pd.concat([pd.read_csv(paths["interim"] / "known_asteroids" / f"known_asteroids__{e}.csv", dtype={"name": str}) for e in frames])
bright = known[known["v_mag"] <= 20.0].groupby("name").agg(v_mag=("v_mag", "mean"), frames=("x", "size"))
bright = bright[bright["frames"] >= 4]
bright["found"] = bright.index.isin(known_tracklets["known_name"])
print(f"\nknown asteroids with V <= 20 in >= 4 frames: {len(bright)}, found as tracklets: {bright['found'].sum()}")
bright.sort_values("v_mag")

## 3. The look-elsewhere effect: why unknown candidates are probably not discoveries
We test thousands of seed combinations per night. Even if each chance alignment is unlikely, *some* will pass every cut.
How many bogus tracklets per night should we expect to pass the tracklet threshold? Use the validation estimate:

In [ ]:
validation_tracklets = pd.concat([pd.read_parquet(paths["processed"] / f"tracklets__{make_unit_tag(unit_of(n))}.parquet")
                                  for n in acquire["nights"]["validation"]])
scores = tracklet_model.predict_proba(np.nan_to_num(validation_tracklets[FEATURE_COLUMNS].to_numpy()))[:, 1]
false_per_night = ((scores >= thresholds["tracklet"]) & (validation_tracklets["label"] == 0)).sum() / len(acquire["nights"]["validation"])
print(f"expected bogus tracklets passing the threshold, per night: {false_per_night:.1f}")
print(f"unknown candidates we actually got tonight: {len(candidates)}")

If the number of candidates is close to the expected number of bogus survivors, most candidates are probably bogus.
That's not failure; it's what honest statistics look like. A real discovery has to survive **vetting**.

## 4. Vetting checklist
For every candidate, a person checks:
1. **Stamps:** a point source in science and difference, nothing in reference, in every frame?
2. **Motion:** steady speed and direction, plausible for an asteroid (main belt near opposition ≈ 25-40 "/hr, westward)?
3. **Brightness:** roughly constant (`mag_std` small)?
4. **Not an artifact:** not along a bad column, not next to a bright star (ghosts, halos), not a satellite streak?
5. **Not known:** no SkyBoT match, even with a looser radius (orbits can be off by more than 4")?
6. **Independent confirmation:** does it show up on *another night*? (The Minor Planet Center needs at least two nights.)

In [ ]:
report = paths["reports"] / night / "index.html"
print("open this report to vet the candidates:", report)
candidates[["tracklet_id", "tracklet_score", "n_detections", "rate_arcsec_per_hour", "mag_mean", "mag_std", "score_min"]].round(3).head(10)

## Exercises
1. Open the report. Classify each candidate with the checklist: real, artifact, or unsure?
2. Loosen `match_radius_pixels` in `crossmatch_known_asteroids` to 10. Do any candidates turn out to be known?
3. Using `rate_arcsec_per_hour` and the direction (`vx`, `vy`), which candidates move like main-belt asteroids?